# 03 · Tool Calling & MCP

**Tool calling**: you describe functions with a **schema**, the model **decides** which one to call, your code **executes** it and returns the result.

Two ways to teach a model to use tools:
1. **Via prompting**: describe the tool in text, ask for a JSON reply.
2. **Via training**: the model emits structured tool calls (`tools=` parameter).

### Choose your LLM backend

- `"ollama"`: a local model via [Ollama](https://ollama.com). Run `ollama pull gemma4:e2b-mlx` first.
- `"api"`: any OpenAI-compatible cloud API (OpenAI, Gemini, Groq, OpenRouter, …). Set `LLM_API_KEY`, `LLM_API_MODEL` and, if not OpenAI, `LLM_API_BASE_URL`.

The rest of the notebook works the same with either backend.

In [1]:
# %pip install -q ollama openai fastmcp
import llm_client as llm

BACKEND = "ollama"  # "ollama" (local) or "api" (OpenAI-compatible cloud API)

llm.configure(BACKEND)
NATIVE_MODEL_ID = llm.current_model()
print(llm.describe())

Backend: Ollama (local)  |  model: gemma4:e2b-mlx


## 1a · Tool calling via prompting

*"Find a bear near me!"*: we describe `find_teddy_bear` in plain text and parse the model's JSON reply.

In [2]:
import json
import re
from typing import Any, Callable, Dict, List, Tuple


def find_teddy_bear(latitude: float, longitude: float) -> Dict[str, Any]:
    """Toy backend — a real one would query a store/inventory API."""
    return {"name": "Teddy", "store": "Teddy's Toy Shop", "distance_km": 1.2, "latitude": latitude + 0.01, "longitude": longitude}


TOOL_PROMPT = """You can call this function:

find_teddy_bear(latitude: float, longitude: float) -> the nearest teddy bear to the given location.

The user's current location is latitude 37.42, longitude -122.17.
If the function helps answer the user, reply with ONLY a JSON object of the form
{"name": "<function name>", "arguments": {...}} and nothing else."""

response = llm.chat(
    model=NATIVE_MODEL_ID,
    messages=[
        {"role": "system", "content": TOOL_PROMPT},
        {"role": "user", "content": "Find a bear near me!"},
    ],
)
raw = response.message.content
print("Raw model output:\n", raw)

Raw model output:
 {"name": "find_teddy_bear", "arguments": {"latitude": 37.42, "longitude": -122.17}}


In [3]:
# We have to parse the call out of free text ourselves — and hope the model followed the format
match = re.search(r"\{.*\}", raw, re.DOTALL)
call = json.loads(match.group(0))
result = find_teddy_bear(**call["arguments"])
print(f"{call['name']}({call['arguments']}) -> {result}")

find_teddy_bear({'latitude': 37.42, 'longitude': -122.17}) -> {'name': 'Teddy', 'store': "Teddy's Toy Shop", 'distance_km': 1.2, 'latitude': 37.43, 'longitude': -122.17}


## 1b · Native tool calling

More robust than prompting:
- Tools are JSON Schema definitions.
- The model returns structured `tool_calls`, so no parsing.
- It can choose among several tools.

In [4]:
# ── Model setup: two toy tools + their JSON Schema definitions ───────────────

def get_weather(city: str) -> str:
    """Toy weather lookup — a real implementation would call a weather API."""
    fake_weather = {"osaka": "28C, sunny", "tokyo": "26C, cloudy", "paris": "19C, rainy"}
    return fake_weather.get(city.lower(), f"No weather data for {city}")


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Toy currency converter — a real implementation would call a live FX API."""
    fake_rates = {("USD", "JPY"): 147.5, ("JPY", "USD"): 1 / 147.5}
    rate = fake_rates.get((from_currency.upper(), to_currency.upper()))
    if rate is None:
        return f"No rate available for {from_currency} -> {to_currency}"
    return f"{amount * rate:.2f} {to_currency.upper()}"


# JSON Schema tool definitions — this is what the model actually sees
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount of money from one currency to another",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount to convert"},
                    "from_currency": {"type": "string", "description": "3-letter source currency code"},
                    "to_currency": {"type": "string", "description": "3-letter target currency code"},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
        },
    },
]

NATIVE_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
}

In [5]:
# ── Live Test 1: a single tool call ───────────────────────────────────────────
response = llm.chat(
    model=NATIVE_MODEL_ID,
    messages=[{"role": "user", "content": "What is the weather in Osaka? Use the tool."}],
    tools=TOOLS_SCHEMA,
)

print("Raw tool_calls from the model:")
print(response.message.tool_calls)

Raw tool_calls from the model:
[ToolCall(function=Function(name='get_weather', arguments={'city': 'Osaka'}))]


In [6]:
response.message.tool_calls

[ToolCall(function=Function(name='get_weather', arguments={'city': 'Osaka'}))]

In [7]:
# Execute whatever the model asked for
for tool_call in response.message.tool_calls:
    fn = NATIVE_TOOL_REGISTRY[tool_call.function.name]
    result = fn(**tool_call.function.arguments)
    print(f"\n{tool_call.function.name}({tool_call.function.arguments}) -> {result}")


get_weather({'city': 'Osaka'}) -> 28C, sunny


In [8]:
# ── Live Test 2: model picks the right tool out of several ───────────────────
for question in [
    "What is the weather in Paris? Use the tool.",
    "Convert 200 USD to JPY. Use the tool.",
    "What's your favorite color?",  # no tool applies — model should just answer
    "What's the weather? Use the tool.",
]:
    response = llm.chat(model=NATIVE_MODEL_ID, messages=[{"role": "user", "content": question}], tools=TOOLS_SCHEMA)
    print(f"Q: {question}")
    if response.message.tool_calls:
        for tool_call in response.message.tool_calls:
            print(f"  -> called {tool_call.function.name}({tool_call.function.arguments})")
    else:
        print(f"  -> no tool call, direct answer: {response.message.content}")
    print()

Q: What is the weather in Paris? Use the tool.
  -> called get_weather({'city': 'Paris'})



Q: Convert 200 USD to JPY. Use the tool.
  -> called convert_currency({'amount': 200, 'from_currency': 'USD', 'to_currency': 'JPY'})



Q: What's your favorite color?
  -> no tool call, direct answer: As an AI, I don't have personal preferences, so I don't have a favorite color.



Q: What's the weather? Use the tool.
  -> no tool call, direct answer: Please tell me which city you are interested in.



### Closing the loop

Append the tool call and a `role="tool"` result, then call the model again for the final answer.

In [9]:
messages = [{"role": "user", "content": "How much is 200 USD in JPY? Use the tool."}]
response = llm.chat(model=NATIVE_MODEL_ID, messages=messages, tools=TOOLS_SCHEMA)
messages.append(response.message)  # 1. the model's tool call

for tool_call in response.message.tool_calls:
    result = NATIVE_TOOL_REGISTRY[tool_call.function.name](**tool_call.function.arguments)  # 2. backend executes it
    print(f"{tool_call.function.name}({tool_call.function.arguments}) -> {result}")
    messages.append({"role": "tool", "content": str(result), "tool_name": tool_call.function.name})

final = llm.chat(model=NATIVE_MODEL_ID, messages=messages, tools=TOOLS_SCHEMA)  # 3. model concludes
print("\nFinal answer:", final.message.content)

convert_currency({'amount': 200, 'from_currency': 'USD', 'to_currency': 'JPY'}) -> 29500.00 JPY



Final answer: 200 USD is equal to 29,500.00 JPY.


### Benefits and challenges

- ✅ More useful models, real-world actions, live data.
- ⚠️ Many tools lower accuracy, schemas cost tokens, and every app re-implements every tool (**N × M** integrations).

## 2 · Model Context Protocol (MCP)

MCP solves N × M: each tool is wrapped **once** as a server; any MCP client discovers and calls it.

`simple_mcp.py` is a tiny [FastMCP](https://gofastmcp.com) server with three file tools (`read_file_tool`, `list_files_tool`, `edit_file_tool`) over HTTP at `http://127.0.0.1:8000/mcp`. The next cell starts it.

> ⚠️ `edit_file_tool` can **write files** on your machine. Run it locally only.

In [10]:
# %pip install -q --force-reinstall --no-cache-dir fastmcp
# If you still get "FastMCP client support is not installed" after this, restart the kernel
# (Kernel > Restart) so it picks up the freshly installed package, then re-run from the top.

In [11]:
# ── Start simple_mcp.py as a background process from this notebook ───────────
# mcp.run(...) blocks forever, so it can't run in a normal cell — that would
# freeze this kernel. Instead we launch it as a subprocess (using this same
# kernel's Python, so it has fastmcp installed) and poll the HTTP port until
# the server is actually accepting connections.
import socket
import subprocess
import sys
import time

MCP_SERVER_SCRIPT = "simple_mcp.py"
MCP_SERVER_HOST, MCP_SERVER_PORT = "127.0.0.1", 8000


def _port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


if _port_open(MCP_SERVER_HOST, MCP_SERVER_PORT):
    print(f"MCP server already running at http://{MCP_SERVER_HOST}:{MCP_SERVER_PORT}/mcp")
else:
    mcp_server_process = subprocess.Popen(
        [sys.executable, MCP_SERVER_SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(30):
        if _port_open(MCP_SERVER_HOST, MCP_SERVER_PORT):
            print(f"MCP server started (pid={mcp_server_process.pid}) at http://{MCP_SERVER_HOST}:{MCP_SERVER_PORT}/mcp")
            break
        if mcp_server_process.poll() is not None:
            print(mcp_server_process.stdout.read())
            raise RuntimeError("simple_mcp.py exited before the server came up")
        time.sleep(0.5)
    else:
        raise TimeoutError("Timed out waiting for the MCP server to start")

MCP server started (pid=61985) at http://127.0.0.1:8000/mcp


In [12]:
# ── Connect to the MCP server over HTTP and discover its tools ───────────────
from fastmcp import Client

# `simple_mcp.py` is a *persistent* server (started by the previous cell or by `python simple_mcp.py`),
# listening at this URL — every client below connects to that one running process rather
# than each spawning its own subprocess (which is what stdio transport would force).
MCP_SERVER_URL = "http://127.0.0.1:8000/mcp"
mcp_client = Client(MCP_SERVER_URL)

async with mcp_client:
    mcp_tools = await mcp_client.list_tools()


def _sanitize_json_schema(schema: Dict[str, Any]) -> Dict[str, Any]:
    """Repair inputSchemas that aren't valid JSON Schema.

    Some MCP servers we don't control (e.g. third-party ones) emit a property's
    "type" as a dict instead of a string/list of strings, which fails Ollama's
    client-side schema validation. Default any such malformed "type" to "object".
    """
    if not isinstance(schema, dict):
        return schema
    fixed = dict(schema)
    if "type" in fixed and not isinstance(fixed["type"], (str, list)):
        fixed["type"] = "object"
    if isinstance(fixed.get("properties"), dict):
        fixed["properties"] = {name: _sanitize_json_schema(prop) for name, prop in fixed["properties"].items()}
    return fixed


# Convert MCP's tool descriptions into the same JSON-Schema shape Ollama expects (see Part 1's TOOLS_SCHEMA)
MCP_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": _sanitize_json_schema(tool.inputSchema),
        },
    }
    for tool in mcp_tools
]

for tool in mcp_tools:
    print(f"{tool.name}: {tool.description}")

read_file_tool: Gets the full content of a file provided by the user.
list_files_tool: Lists the files in a directory provided by the user.
edit_file_tool: Replaces first occurrence of old_str with new_str in file. If old_str is empty, creates/overwrites file with new_str.


In [13]:
# ── Live Test: model picks a tool, we dispatch the call to the MCP server ────
MCP_MODEL_ID = NATIVE_MODEL_ID


async def call_mcp_tool(name: str, arguments: Dict[str, Any]) -> str:
    """Run a single MCP tool call as an RPC against the SimpleMCPTestServer subprocess."""
    async with mcp_client:
        result = await mcp_client.call_tool(name, arguments)
    return result.content[0].text  # JSON string the tool returned


response = llm.chat(
    model=MCP_MODEL_ID,
    messages=[{"role": "user", "content": "List the files in the current directory ('.')."}],
    tools=MCP_TOOLS_SCHEMA,
)

In [14]:
print("Raw tool_calls from the model:")
print(response.message.tool_calls)

Raw tool_calls from the model:
[ToolCall(function=Function(name='list_files_tool', arguments={'path': '.'}))]


In [15]:
for tool_call in response.message.tool_calls:
    result = await call_mcp_tool(tool_call.function.name, tool_call.function.arguments)
    print(f"\n{tool_call.function.name}({tool_call.function.arguments}) -> {result}")


list_files_tool({'path': '.'}) -> {"path":"/Users/rinabuoy/Desktop/Courses/TP2.0/notebooks","files":[{"filename":"01_llm_basics.ipynb","type":"file"},{"filename":"02_rag.ipynb","type":"file"},{"filename":"03_tool_calling_and_mcp.ipynb","type":"file"},{"filename":"__pycache__","type":"dir"},{"filename":"simple_mcp.py","type":"file"},{"filename":"rag_utils.py","type":"file"},{"filename":"llm_client.py","type":"file"},{"filename":"04_ai_agents.ipynb","type":"file"}]}


In [16]:
# ── Agent loop over MCP tools — same shape as Part 1's tool loop ─────────────
async def run_mcp_agent(user_message: str, max_steps: int = 4, model: str = MCP_MODEL_ID) -> str:
    messages = [{"role": "user", "content": user_message}]
    for step in range(max_steps):
        response = llm.chat(model=model, messages=messages, tools=MCP_TOOLS_SCHEMA)
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            result = await call_mcp_tool(tool_call.function.name, tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "content": result, "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."


SAMPLE_MODULE_PATH = "simple_mcp.py"  # any file in the current directory works
answer = await run_mcp_agent(
    f"List the files in the current directory ('.'), then read {SAMPLE_MODULE_PATH} and summarize what it contains."
)
print(f"\nFinal answer: {answer}")

  [step 1] list_files_tool({'path': '.'}) -> {"path":"/Users/rinabuoy/Desktop/Courses/TP2.0/notebooks","files":[{"filename":"01_llm_basics.ipynb","type":"file"},{"filename":"02_rag.ipynb","type":"file"},{"filename":"03_tool_calling_and_mcp.ipynb","type":"file"},{"filename":"__pycache__","type":"dir"},{"filename":"simple_mcp.py","type":"file"},{"filename":"rag_utils.py","type":"file"},{"filename":"llm_client.py","type":"file"},{"filename":"04_ai_agents.ipynb","type":"file"}]}


  [step 2] read_file_tool({'filename': 'simple_mcp.py'}) -> {"file_path":"/Users/rinabuoy/Desktop/Courses/TP2.0/notebooks/simple_mcp.py","content":"from pathlib import Path\nfrom typing import Any, Dict, List\nfrom fastmcp import FastMCP\n\nmcp = FastMCP(name=\"SimpleMCPTestServer\")\n\n\ndef resolve_abs_path(path_str: str) -> Path:\n    \"\"\"\n    file.py -> /Users/home/mihail/modern-software-dev-lectures/file.py\n    \"\"\"\n    path = Path(path_str).expanduser()\n    if not path.is_absolute():\n        path = (Path.cwd() / path).resolve()\n    return path\n\n@mcp.tool\ndef read_file_tool(filename: str) -> Dict[str, Any]:\n    \"\"\"\n    Gets the full content of a file provided by the user.\n    :param filename: The name of the file to read.\n    :return: The full content of the file.\n    \"\"\"\n    full_path = resolve_abs_path(filename)\n    print(full_path)\n    # TODO (mihail): Be more defensive in the file reading here\n    with open(str(full_path), \"r\") as f:\n        cont


Final answer: The file listing showed the following files in the current directory:
*   `01_llm_basics.ipynb`
*   `02_rag.ipynb`
*   `03_tool_calling_and_mcp.ipynb`
*   `__pycache__` (directory)
*   `simple_mcp.py`
*   `rag_utils.py`
*   `llm_client.py`
*   `04_ai_agents.ipynb`

The content of `simple_mcp.py` is a Python script that sets up a local server using the `FastMCP` library. It defines several tools that can be exposed to an external system (like an LLM agent) for interacting with the local file system and file manipulation:

1.  **`resolve_abs_path(path_str)`**: A utility function to correctly resolve file paths, handling relative paths by resolving them against the current working directory or making them absolute.
2.  **`read_file_tool(filename)`**: A tool that, when called, retrieves the full content of a specified file.
3.  **`list_files_tool(path)`**: A tool that lists the files and directories within a given path.
4.  **`edit_file_tool(path, old_str, new_str)`**: A too

### A second, remote MCP server

The same client code works with a remote server: a [Gradio Space](https://huggingface.co/spaces) at `https://rinabuoy-egd-mcp.hf.space/gradio_api/mcp/`. One agent loop routes each call to the server that owns the tool.

In [17]:
# ── Connect to a second, remote MCP server (a Gradio Space) ──────────────────
REMOTE_MCP_SERVER_URL = "https://rinabuoy-egd-mcp.hf.space/gradio_api/mcp/"
remote_mcp_client = Client(REMOTE_MCP_SERVER_URL)

async with remote_mcp_client:
    remote_mcp_tools = await remote_mcp_client.list_tools()

for tool in remote_mcp_tools:
    print(f"{tool.name}: {tool.description}")

EGD_MCP_search_graph: Search graph nodes whose string properties contain a keyword. Performs a case-insensitive substring search across every string-typed property of every node (numeric, list and embedding properties are skipped), optionally restricted to a single label. Useful as a general purpose entity lookup over the knowledge graph. Returns: Numbered, citation-ready text blocks (title, source, tags, content), ready to paste into an LLM's context window, or "No results found."
EGD_MCP_hybrid_search: Hybrid vector + graph search over the knowledge graph. Embeds the query with BAAI/bge-small-en-v1.5 and finds the most semantically similar Page nodes using the Neo4j vector index, then expands each match through the graph to surface directly connected context (tags, linked pages, related entities). This combines the strengths of vector similarity (semantic relevance) with graph structure (explicit relationships) and is generally the best default choice for retrieval-augmented generati

In [18]:
# ── Combine both servers into one tools= list, tracking which client owns each tool ─
TOOL_NAME_TO_CLIENT: Dict[str, Client] = {tool.name: mcp_client for tool in mcp_tools}
TOOL_NAME_TO_CLIENT.update({tool.name: remote_mcp_client for tool in remote_mcp_tools})

COMBINED_TOOLS_SCHEMA = MCP_TOOLS_SCHEMA + [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": _sanitize_json_schema(tool.inputSchema),
        },
    }
    for tool in remote_mcp_tools
]


async def call_any_mcp_tool(name: str, arguments: Dict[str, Any]) -> str:
    """Dispatch a tool call to whichever MCP server (local or remote) actually owns it."""
    client = TOOL_NAME_TO_CLIENT[name]
    async with client:
        result = await client.call_tool(name, arguments)
    return result.content[0].text

In [19]:
# ── Agent loop spanning both MCP servers ──────────────────────────────────────
async def run_multi_mcp_agent(user_message: str, max_steps: int = 4, model: str = MCP_MODEL_ID) -> str:
    messages = [{"role": "user", "content": user_message}]
    for step in range(max_steps):
        response = llm.chat(model=model, messages=messages, tools=COMBINED_TOOLS_SCHEMA)
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            result = await call_any_mcp_tool(tool_call.function.name, tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "content": result, "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."


# Start with a question the local server can answer; once you've seen the remote
# server's tool names/descriptions printed above, extend this to exercise those too.
answer = await run_multi_mcp_agent(
    f"List the files in the current directory ('.'), then read {SAMPLE_MODULE_PATH} and summarize what it contains."
)
print(f"\nFinal answer: {answer}")

  [step 1] list_files_tool({'path': '.'}) -> {"path":"/Users/rinabuoy/Desktop/Courses/TP2.0/notebooks","files":[{"filename":"01_llm_basics.ipynb","type":"file"},{"filename":"02_rag.ipynb","type":"file"},{"filename":"03_tool_calling_and_mcp.ipynb","type":"file"},{"filename":"__pycache__","type":"dir"},{"filename":"simple_mcp.py","type":"file"},{"filename":"rag_utils.py","type":"file"},{"filename":"llm_client.py","type":"file"},{"filename":"04_ai_agents.ipynb","type":"file"}]}


  [step 2] read_file_tool({'filename': 'simple_mcp.py'}) -> {"file_path":"/Users/rinabuoy/Desktop/Courses/TP2.0/notebooks/simple_mcp.py","content":"from pathlib import Path\nfrom typing import Any, Dict, List\nfrom fastmcp import FastMCP\n\nmcp = FastMCP(name=\"SimpleMCPTestServer\")\n\n\ndef resolve_abs_path(path_str: str) -> Path:\n    \"\"\"\n    file.py -> /Users/home/mihail/modern-software-dev-lectures/file.py\n    \"\"\"\n    path = Path(path_str).expanduser()\n    if not path.is_absolute():\n        path = (Path.cwd() / path).resolve()\n    return path\n\n@mcp.tool\ndef read_file_tool(filename: str) -> Dict[str, Any]:\n    \"\"\"\n    Gets the full content of a file provided by the user.\n    :param filename: The name of the file to read.\n    :return: The full content of the file.\n    \"\"\"\n    full_path = resolve_abs_path(filename)\n    print(full_path)\n    # TODO (mihail): Be more defensive in the file reading here\n    with open(str(full_path), \"r\") as f:\n        cont


Final answer: The file `simple_mcp.py` is a Python script that defines and runs a simple test server using the `FastMCP` framework.

**Key functionalities provided by the script:**

1.  **`resolve_abs_path(path_str)`**: A utility function to correctly resolve file paths, handling both absolute and relative paths, ensuring they are absolute paths.
2.  **`read_file_tool(filename)`**: A tool that allows users to retrieve the complete content of a specified file.
3.  **`list_files_tool(path)`**: A tool that allows users to list the files and directories within a given directory path.
4.  **`edit_file_tool(path, old_str, new_str)`**: A tool that enables file editing by replacing a specific string in a file, or overwriting the file if no string is provided for replacement.

The script sets up an HTTP server running on `http://127.0.0.1:8000/mcp` to expose these tools, allowing for file reading, listing, and editing operations.


In [20]:
# ── Clean up: stop the local MCP server if this notebook started it ──────────
if "mcp_server_process" in globals() and mcp_server_process.poll() is None:
    mcp_server_process.terminate()
    print("MCP server stopped")

MCP server stopped


## Key takeaways

- Tool calling: the model picks a function and arguments, your code runs it.
- Native tool calling (`tools=` + JSON Schema) beats prompt-based parsing.
- MCP standardizes tools as servers, local or remote, reusable by any app.